In [1]:
# ════════════════════════════════════════════════════════════════
# compare_mel.py — Compare spectrogramme Python vs ATOMS3R
# ════════════════════════════════════════════════════════════════
import librosa
import librosa.filters
import numpy as np
from pathlib import Path

AUDIO_PATH = Path("C:/Users/asalou/S7/stages7/project/wav_audio/Audio_3.wav")
SEG_IDX    = 25  # ← segment miction fort (score 1.000 sur PC)
SEG_LEN    = 8000
N_MELS, N_FFT, HOP_MEL = 64, 512, 160

y, sr = librosa.load(AUDIO_PATH, sr=16000, mono=True)
y = y / np.max(np.abs(y))

seg = y[SEG_IDX*SEG_LEN:(SEG_IDX+1)*SEG_LEN]

# ── Méthode librosa exacte ────────────────────────────────────
mel = librosa.feature.melspectrogram(
    y=seg.astype(float), sr=16000,
    n_mels=N_MELS, n_fft=N_FFT,
    hop_length=HOP_MEL, fmax=6000,
    power=2.0)
mel_db = librosa.power_to_db(mel, ref=np.max)
mn, mx = mel_db.min(), mel_db.max()
mel_norm = (mel_db - mn) / (mx - mn)

print(f"Librosa — min={mel_norm.min():.3f} mean={mel_norm.mean():.3f} max={mel_norm.max():.3f}")
print(f"Librosa — mel shape: {mel_norm.shape}")

# Afficher les 5 premières valeurs de la première bande
print(f"Librosa mel[0][:5] = {mel_norm[0,:5].round(3)}")
print(f"Librosa mel[32][:5] = {mel_norm[32,:5].round(3)}")

# ── Vérifier les paramètres ───────────────────────────────────
import librosa.filters
fb = librosa.filters.mel(sr=16000, n_fft=N_FFT, n_mels=N_MELS, fmax=6000)
print(f"\nFilterbank librosa shape: {fb.shape}")
print(f"Filterbank sum par bande: {fb.sum(axis=1)[:5].round(4)}")

# ── Reconstruire manuellement comme votre C++ ─────────────────
frames = librosa.util.frame(seg, frame_length=N_FFT, hop_length=HOP_MEL)
window = np.hanning(N_FFT)  # ← symmetric
# vs periodic :
window_periodic = 0.5*(1 - np.cos(2*np.pi*np.arange(N_FFT)/N_FFT))

print(f"\nHann symmetric[255] = {window[255]:.6f}")
print(f"Hann periodic[255]  = {window_periodic[255]:.6f}")
print(f"Différence max = {np.max(np.abs(window - window_periodic)):.6f}")

# Quel type de fenêtre utilise librosa ?
import scipy.signal
win_librosa = scipy.signal.get_window('hann', N_FFT, fftbins=True)
print(f"\nLibrosa utilise fftbins=True (periodic)")
print(f"win_librosa[255] = {win_librosa[255]:.6f}")
print(f"= periodic ? {np.allclose(win_librosa, window_periodic)}")

Librosa — min=0.000 mean=0.504 max=1.000
Librosa — mel shape: (64, 51)
Librosa mel[0][:5] = [0.527 0.641 0.671 0.678 0.694]
Librosa mel[32][:5] = [0.281 0.457 0.4   0.349 0.373]

Filterbank librosa shape: (64, 257)
Filterbank sum par bande: [0.0299 0.0306 0.0352 0.0299 0.0314]

Hann symmetric[255] = 0.999991
Hann periodic[255]  = 0.999962
Différence max = 0.004706

Librosa utilise fftbins=True (periodic)
win_librosa[255] = 0.999962
= periodic ? True


In [2]:
# Ajoutez dans compare_mel.py
import librosa.filters

# Avec norm='slaney' (défaut librosa)
fb_slaney = librosa.filters.mel(
    sr=16000, n_fft=512, n_mels=64, fmax=6000,
    norm='slaney')

# Sans normalisation
fb_none = librosa.filters.mel(
    sr=16000, n_fft=512, n_mels=64, fmax=6000,
    norm=None)

print(f"Avec norm='slaney' sum[:3] = {fb_slaney.sum(axis=1)[:3].round(4)}")
print(f"Sans norm sum[:3]          = {fb_none.sum(axis=1)[:3].round(4)}")
print(f"Ratio slaney/none          = {(fb_slaney.sum(axis=1)/fb_none.sum(axis=1))[:3].round(4)}")

Avec norm='slaney' sum[:3] = [0.0299 0.0306 0.0352]
Sans norm sum[:3]          = [1.258  1.2898 1.4841]
Ratio slaney/none          = [0.0237 0.0237 0.0237]


In [3]:
# test_pipeline.py
import sys
sys.path.append("C:/Users/asalou/S7/stages7/project/projet_ihm")
from pipeline import charger_modele, analyser_audio
from pathlib import Path

MODEL_PATH = "C:/Users/asalou/S7/stages7/project/augmentation/aug/cnn2d_v2.keras"
AUDIO_PATH = "C:/Users/asalou/S7/stages7/project/wav_audio/Audio_3.wav"

model, le = charger_modele(MODEL_PATH)
result = analyser_audio(AUDIO_PATH, model, le)

print(f"Durée miction  : {result['duree_miction_s']}s")
print(f"Type           : {result['type_miction']}")
print(f"N segments     : {result['n_segments']}")
print(f"Chasse         : {result['chasse_detectee']}")
print(f"\nLabels bruts (avant correction) :")
# Compter miction
labels = result['labels_pred']
n_mic = labels.count('miction_active')
print(f"miction_active : {n_mic} segments = {n_mic*0.25:.2f}s")
print(f"bruit_chasse   : {labels.count('bruit_chasse')} segments")
print(f"bruit_ambiant  : {labels.count('bruit_ambiant')} segments")

 Modèle chargé — classes : ['bruit_ambiant' 'bruit_chasse' 'miction_active']
Signal : 493568 samples = 30.8s  max=1.0000
Segments : 122
X_mel : (122, 64, 51, 1)
Durée miction  : 8.25s
Type           : jet_continu
N segments     : 122
Chasse         : True

Labels bruts (avant correction) :
miction_active : 33 segments = 8.25s
bruit_chasse   : 27 segments
bruit_ambiant  : 62 segments
